In [1]:
import sys
sys.path.insert(0, "../")

from all_type_parser.all_type_parser import parse_folder

FOLDER = "../../data/failed_parse_examples"
saved = parse_folder(FOLDER)

[all_type_parser] found 5 file(s) in ../../data/failed_parse_examples

[all_type_parser] [1/5] 451_Application overview - UKRI Funding Service (7).pdf


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

[all_type_parser] all PDF parsers returned empty — falling back to document_parser


Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 flo

[all_type_parser] ✓ document_parser + all_other_parser succeeded
[all_type_parser] saved → ..\..\data\failed_parse_examples\json_data\451_Application overview - UKRI Funding Service (7).json

[all_type_parser] [2/5] 559_NIHR513015 Abraham Contreras - Full Application - final.pdf
[all_type_parser] ✓ pdf_parser succeeded
[all_type_parser] saved → ..\..\data\failed_parse_examples\json_data\559_NIHR513015 Abraham Contreras - Full Application - final.json

[all_type_parser] [3/5] 867_NIHR515958 Katya Herman - Full Application 06.03.pdf
[all_type_parser] ✓ pdf_parser succeeded
[all_type_parser] saved → ..\..\data\failed_parse_examples\json_data\867_NIHR515958 Katya Herman - Full Application 06.03.json

[all_type_parser] [4/5] 925_NIHR516523 Claire Higgins - Full Application.pdf
[all_type_parser] ✓ pdf_parser succeeded
[all_type_parser] saved → ..\..\data\failed_parse_examples\json_data\925_NIHR516523 Claire Higgins - Full Application.json

[all_type_parser] [5/5] IC00011-NIHR503119 Lisa Edel

In [6]:
import sys, importlib, json
sys.path.insert(0, "../")

import all_type_parser.RfPB_parser as rfpb
importlib.reload(rfpb)

PDF = "../../data/all_types_examples/IC00029_RfPB.pdf"

# Debug: check what raw text is extracted from the CV section
raw_cv = rfpb._get_section_raw_text(PDF, rfpb.SECTION_CV_LEAD)
print("=== Raw CV - Lead Applicant(s) text ===")
print(raw_cv[:1000])
print()

# Full parse result
result = rfpb.extract_all_sections(PDF)
team = result.get("LEAD APPLICANT & RESEARCH TEAM", {})
print("=== LEAD APPLICANT & RESEARCH TEAM ===")
print(json.dumps(team, indent=2))


=== Raw CV - Lead Applicant(s) text ===
RfPB URD & Specialisms highlight notice: Allied Health Professionals (Stage 2)
A flow diagram illustrating the study design / flow of participants (document upload), if appropriate
Confirmed
A full and accurate detailed budget breakdown
Confirmed
A clear justification of costs / value for money
Confirmed
n
References (document upload)
o
Confirmed
i
A clear Detailed Research Plan outlining the study design, methods, dsissemination etc.
Confirmed
s
A CTU letter of support if required (document upload) i
m
Confirmed
Completed Schedule of Events Cost Attribubtion Tool (SoECAT)
Confirmed
u
14. CV - Lead Applicant(s)
s
Full Name Dr GRAEME O'CONNOR Position Paediatric Dietitian
-
Department Institution Great Ormond Street
eIntensive Care Hospital for Children NHS
Foundation Trust
ORCID iD r0000-0001-8625-9264 Telephone No. +44 7958543828
Address Line 1 Level 2 Nurses Building Country United Kingdom
P
Address Line 2 Great Ormond Street Postcode WC1N 3JH


In [5]:
import sys, importlib, pdfplumber, re
sys.path.insert(0, "../")
import all_type_parser.RfPB_parser as rfpb
importlib.reload(rfpb)

PDF = "../../data/all_types_examples/IC00029_RfPB.pdf"
TARGET = rfpb.SECTION_CV_LEAD  # "CV - Lead Applicant(s)"

with pdfplumber.open(PDF) as pdf:
    for pno, page in enumerate(pdf.pages):
        boxes = rfpb._page_section_boxes(page)
        if not boxes:
            continue
        words = page.extract_words(x_tolerance=1, y_tolerance=3)
        headings_found = []
        for box in boxes:
            box_words = sorted(
                [w for w in words if w["top"] <= box["bottom"] and w["bottom"] >= box["top"]],
                key=lambda w: w["x0"],
            )
            h = re.sub(r'\s+', ' ', rfpb._strip_number(" ".join(w["text"] for w in box_words).strip()))
            if h:
                headings_found.append(h)
        # Only print pages near the CV section
        if any("CV" in h for h in headings_found) or (pno >= 38 and pno <= 43):
            print(f"Page {pno+1}: headings_found = {headings_found}")
            print(f"  -> TARGET '{TARGET}' in list: {TARGET in headings_found}")
            print()


Page 39: headings_found = ['Administrative contact details']
  -> TARGET 'CV - Lead Applicant(s)' in list: False

Page 40: headings_found = ['Research & Development office contact', 'Acknowledgement, review and submit s i o n']
  -> TARGET 'CV - Lead Applicant(s)' in list: False

Page 41: headings_found = ['CV - Lead Applicant(s) s u Attribubtion m']
  -> TARGET 'CV - Lead Applicant(s)' in list: False

Page 43: headings_found = ['o nRole', 'CV - Co-applicants s i o nRole', 'PublicaPtions']
  -> TARGET 'CV - Lead Applicant(s)' in list: False

Page 44: headings_found = []
  -> TARGET 'CV - Lead Applicant(s)' in list: False



In [3]:
import sys, importlib
sys.path.insert(0, "../")

import all_type_parser.RfPB_parser as rfpb
importlib.reload(rfpb)

PDF = "../../data/all_types_examples/IC00029_RfPB.pdf"

# List all detected section headings (as seen by the parser)
lines = rfpb.extract_lines_pdfplumber(PDF)
lines = rfpb.filter_rfpb_lines(lines)
titles = rfpb.list_section_titles(lines)
print("Detected section headings:")
for t in titles:
    print(repr(t))


Detected section headings:
'Table Of Contents'
'1. The Research Team'
'u'
's'
'-'
'e'
'2. History of application'
'3. Scientific abstract'
'4. Plain English Summary'
'5. Changes from first stage'
'NIHRCC'
'Dr Graeme O’Connor'
'Grange House'
'By Email'
'15 Church Street'
'Twickenham'
'TW1 3NL'
'Tel: 020 8843 8000'
'Email: urdrfpb@nihr.ac.uk'
'www.nihr.ac.uk'
'Wednesday 28 August 2024 n'
"Dear Dr O'Connor,"
'o'
'RfPB Under-represented disciplines and specialisms highlight notice: Allied Health'
'i'
'Professionals: NIHR208140 - Single-test accuracy study to s assess the diagnostic'
'value of faecal calprotectin to diagnose necrotising enterocolitis in infants with heart'
'defects s'
'i'
'Following the Stage 1 review of applications submitted to the above competition, I am'
'pleased to inform you that your application m has been invited to progress to Stage 2 of the'
'application process.'
'b'
'Further to our previous correspondence, I include your feedback from the Committee'
'Assessors b